In [22]:
from dotenv import load_dotenv
import os


from api_requests.prompter import DeepResearchGemini
from params.paths import ROOT_DIR
from dbio.representative_db import connect_db, create_tables_if_not_exist, Person, ElectionResult, iterate_all_persons, get_election_result_by_person_id, insert_x_account, get_x_account_by_person_id

prompter = DeepResearchGemini()


x_account_collection_sys_prompt = """\\
You are a research assistant that helps users find X account information of politicians. I am going to provide you with the name and election details of a politician, and you will search for their X account. If you find it, please provide the account handle (e.g., @example). If you cannot find the account, please respond with "Not found". Make sure to only provide the account handle or "Not found" as your response. Do not include any additional information or explanations. Make sure to always use Google Search to find the information.
    
    """

x_account_collection_prompt = lambda name_kanji, election_details: f"""\\
Here is the politician's information:
Name: \n{name_kanji}\n\n
Election Details: \n{election_details}
Please find their X account.**ALWAYS USE GOOGLE SEARCH WHEN ANSWERING. 
"""
load_dotenv()



conn = connect_db(
    dbname="kokkaidoc",
    user="postgres",
    password=os.getenv("PSQL_DATABASE_PASSWORD"),
    host="localhost",
    port=5432,
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [23]:
from googleapiclient.discovery import build
from dotenv import load_dotenv
import os
import time
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
SEARCH_ENGINE_ID = os.getenv("SEARCH_ENGINE_ID")

def google_search(query: str) -> list[str]:
	service = build("customsearch", "v1", developerKey=GOOGLE_API_KEY)
	results = service.cse().list(q=query, cx=SEARCH_ENGINE_ID).execute()
	links = []
	if 'items' in results:
		for item in results['items']:
			links.append(item['link'])
	return links

# Example usage:
links = google_search("masayo_tanabu")
print(links)

['https://en.wikipedia.org/wiki/Masayo_Tanabu', 'https://x.com/masayo_tanabu?lang=en', 'https://fr.wikipedia.org/wiki/Masayo_Tanabu', 'https://www.dpfp.or.jp/download/49137.pdf', 'https://www.asahi.com/ajw/articles/14497158', 'https://twitter.com/preciousyuumin/status/1555576497901281283', 'https://en.namu.wiki/w/%EB%AF%BC%EC%82%AC%ED%98%91%ED%9A%8C', 'https://english.kyodonews.net/articles/-/39678', 'https://www.punjabnewsexpress.com/news/news/japans-upper-house-expels-absentee-lawmaker-202951', 'https://en.namu.wiki/w/%EB%A7%88%ED%82%A4%EC%95%BC%EB%A7%88%20%ED%9E%88%EB%A1%9C%EC%97%90']


In [37]:
def check_x_account_exists(account_handle: str) -> bool:
    links = google_search(f'site:x.com {account_handle}')
    print("Got links to check existence:", links)
    for link in links:
        if link.lower().endswith('x.com/' + account_handle.lower().lstrip('@')):
            return True
    return False
    
with conn.cursor() as cur:
    create_tables_if_not_exist(cur)
    latest_person_id = None
    
    cur.execute("SELECT MAX(person_id) FROM x_account;")
    row = cur.fetchone()
    if row is not None and row[0] is not None:
        latest_person_id = row[0]
        print(f"Latest person_id in person_x_account: {latest_person_id}")
    done_persons = set()
    with open(os.path.join(ROOT_DIR, 'logs', 'x_account_collection.log'), 'r') as log_file:
        for line in log_file:
            done_persons.add(line.strip())

    person_after_2000 = []
    for person in iterate_all_persons(cur):
        election_result = get_election_result_by_person_id(cur, person.person_id)
        last_election_year = max([e.election_date.year for e in election_result]) if election_result else None
        if last_election_year is not None and last_election_year >= 2008:
            person_after_2000.append(person)

    
    person_after_2000 = [p for p in person_after_2000 if (p.name_kanji not in done_persons)]



    for person in person_after_2000:
        print(f"Processing {person.name_kanji} (ID: {person.person_id}) Progress: {person_after_2000.index(person)+1}/{len(person_after_2000)} ")
        if person.name_kanji in done_persons:
            print(f"Already processed {person.name_kanji}. Skipping.")
            continue
        with open(os.path.join(ROOT_DIR, 'logs', 'x_account_collection.log'), 'a') as log_file:
            log_file.write(person.name_kanji + "\n")
        # if latest_person_id is not None and person.person_id <= latest_person_id:
        #     continue  # Skip already processed persons
        if get_x_account_by_person_id(cur, person.person_id) is not None:
            print(f"Person ID {person.person_id} already has X account. Skipping.")
            continue  # Skip if already has x_account
        election_result = get_election_result_by_person_id(cur, person.person_id)
        last_election_year = max([e.election_date.year for e in election_result]) if election_result else None
        print(last_election_year, person.name_kanji)
        election_result_str = "\n".join([str(er) for er in election_result])
        prompt = x_account_collection_prompt(person.name_kanji, election_result_str)
        count = 0
        while True: 
            resp, cleaned_supports, cleaned_chunks = prompter.prompt(prompt=prompt, system_prompt=x_account_collection_sys_prompt)
            try:
                if resp.strip() != "Not found":
                    if not check_x_account_exists(resp.replace("@", "").strip()):
                        print(f"Account {resp.strip()} does not seem to exist. Treating as Not found.")
                        x_account = ""
                    else:
                        x_account = resp.replace("@", "").strip()
                        print(f"Found X account: {x_account} for {person.name_kanji}")
                        break
                count += 1
                if count >= 3:
                    x_account = ""
                    print(f"Could not find valid X account for {person.name_kanji} after 3 attempts. Marking as Not found.")
                    break
            except Exception as e:
                print(f"Error while checking X account for {person.name_kanji}: {e}")
                x_account = ""
                break
  
        insert_x_account(cur, person.person_id, x_account)
        conn.commit()
        time.sleep(2)  # To avoid hitting rate limits
         


Latest person_id in person_x_account: 5728
Processing 小池正昭 (ID: 2749) Progress: 1/695 
Person ID 2749 already has X account. Skipping.
Processing 三ツ矢憲生 (ID: 2751) Progress: 2/695 
2017 三ツ矢憲生
---------------------------------------------------------------
PROMPTING GEMINI
\
You are a research assistant that helps users find X account information of politicians. I am going to provide you with the name and election details of a politician, and you will search for their X account. If you find it, please provide the account handle (e.g., @example). If you cannot find the account, please respond with "Not found". Make sure to only provide the account handle or "Not found" as your response. Do not include any additional information or explanations. Make sure to always use Google Search to find the information.

    


\
Here is the politician's information:
Name: 
三ツ矢憲生


Election Details: 
Election date: 2003-11-09 Election name: 第43回衆議院議員総選挙 District: 三重5区 Party: 自由民主党 Result: 当選 (1)
Electi

KeyboardInterrupt: 

## Merging with the manually collected data

In [19]:
from params.paths import ROOT_DIR
import os
import pandas as pd
from dbio.representative_db import connect_db, get_person_by_column, get_by_x_account	
from dotenv import load_dotenv
load_dotenv()
manual_x_account_dir = os.path.join(ROOT_DIR, "data", "data_x_account_manual")
print(os.listdir(manual_x_account_dir))


keys = {
	"参議院_2022,2025当選者 - 統合版.csv":{
		"hiragana_name": "候補者名(ひらがな）",
		"x_account": "ID 番号_1",
	},
	"衆議院2024年選挙結果とXアカウント - 統合版.csv":{
		"hiragana_name": "候補者名(ひらがな）",
		"x_account": "ID 番号_1",
	},
	"衆議院2021年選挙結果とXアカウント - 統合版.csv":{
		"hiragana_name": "候補者名(ひらがな）",
		"x_account": "ID 番号",
	}
}


def iterate_manual_x_account_data()-> tuple[str, str]:
	for filename, key in keys.items():
		filepath = os.path.join(manual_x_account_dir, filename)
		df = pd.read_csv(filepath)
		print(f"Processing {filename} with {len(df)} records")
		# with conn.cursor() as cur:
		for idx, row in df.iterrows():
			hiragana_name = row[key["hiragana_name"]]
			x_account = row[key["x_account"]]
			if hiragana_name and bool(x_account) and str(x_account).strip() != "nan":
				try:
					hiragana_name = hiragana_name.replace("　", "").strip().replace(" ", "")
					x_account = str(x_account).replace("@", "").strip()
					yield hiragana_name, x_account

				except Exception as e:
					print(row)
					break

conn = connect_db(
    dbname="kokkaidoc",
    user="postgres",
    password=os.getenv("PSQL_DATABASE_PASSWORD"),
    host="localhost",
    port=5432,
)

non_existent_names = []


with conn.cursor() as cur:
	for hiragana_name, x_account in iterate_manual_x_account_data():
		# get person id with hiragana
		if get_by_x_account(cur, x_account):
			print(f"X account {x_account} already exists in DB. Skipping.")
			continue
		person = get_person_by_column(cur, "name_kana", hiragana_name)
		if len(person) > 1:
			print(f"Multiple persons found for {hiragana_name} x_account={x_account}:")

			for p in person:
				print(f"Person ID: {p.person_id}, Name Kanji: {p.name_kanji}")
				print("Election results:")
				results = get_election_result_by_person_id(cur, p.person_id)
				for r in results:
					print(f"  {r}")
			index = input(f"Enter the index of the correct person for {hiragana_name} (or 's' to skip): ")
			if not index:
				continue
			else:
				index = int(index)
				person = person[index]
				print(f"Updating X account for {hiragana_name} (person_id={person.person_id}, name_kanji={person.name_kanji}) to {x_account}")
				insert_x_account(cur, person.person_id, x_account)

		elif len(person) == 1:
			person = person[0]
			print(f"Updating X account for {hiragana_name} (person_id={person.person_id}, name_kanji={person.name_kanji}) to {x_account}")
			insert_x_account(cur, person.person_id, x_account)
			conn.commit()

		else:
			print(f"No person found for hiragana name: {hiragana_name}")
			non_existent_names.append(hiragana_name)



print("Non-existent names:")
for name in non_existent_names:
    print(name)

['衆議院2021年選挙結果とXアカウント - 統合版.csv', '参議院_2022,2025当選者 - 統合版.csv', '衆議院2024年選挙結果とXアカウント - 統合版.csv']
Processing 参議院_2022,2025当選者 - 統合版.csv with 248 records
X account eri_line already exists in DB. Skipping.
X account gaku_hasegawa already exists in DB. Skipping.
X account t_funahashi already exists in DB. Skipping.
X account iwamoto_tsuyo already exists in DB. Skipping.
X account katsubekenji already exists in DB. Skipping.
X account harumi_takahasi already exists in DB. Skipping.
X account masayo_tanabu already exists in DB. Skipping.
X account ekidoguchi already exists in DB. Skipping.
X account TeamYokosawa already exists in DB. Skipping.
X account DrSakurai already exists in DB. Skipping.
X account norinotes already exists in DB. Skipping.
X account hirooishii6 already exists in DB. Skipping.
X account teratashizuka already exists in DB. Skipping.
X account yasue_funayama0 already exists in DB. Skipping.
X account hagamichiya already exists in DB. Skipping.
X account hoshihokuto_web al

In [21]:
def iterate_all_x_account_data()-> tuple[int, str]:
	conn = connect_db(
		dbname="kokkaidoc",
		user="postgres",
		password=os.getenv("PSQL_DATABASE_PASSWORD"),
		host="localhost",
		port=5432,
	)
	with conn.cursor() as cur:
		cur.execute("SELECT person_id, x_account FROM x_account;")
		rows = cur.fetchall()
		for row in rows:
			yield row[0], row[1]

non_existent_accounts = []

for person_id, x_account in iterate_all_x_account_data():
	# print(f"Person ID: {person_id}, X Account: {x_account}")
	if not check_x_account_exists(x_account):
		print(f"X account {x_account} for person_id {person_id} does not exist.")
		non_existent_accounts.append((person_id, x_account))
	time.sleep(3)

with open("non_existent_x_accounts.txt", "w") as f:
	for person_id, x_account in non_existent_accounts:
		f.write(f"Person ID: {person_id}, X Account: {x_account}\n")
print(f"Total non-existent X accounts: {len(non_existent_accounts)}")

Got links to check existence: ['https://x.com/kokutakeiji', 'https://x.com/kokutakeiji/status/1963814215569011089', 'https://x.com/kokutakeiji/status/1946214768441770481', 'https://x.com/kokutakeiji/status/1924310662924452064', 'https://x.com/kokutakeiji/status/1923171803897245930', 'https://x.com/kokutakeiji/status/1831846456904622409', 'https://x.com/kokutakeiji/status/1897289746109620379', 'https://x.com/kokutakeiji/status/1787289301388259564', 'https://x.com/kokutakeiji/status/1932232891750388042', 'https://x.com/kokutakeiji/status/1930846586424422807']
X account (5,kokutakeiji) for person_id 5 does not exist.
Got links to check existence: ['https://x.com/83Weeks', 'https://x.com/DasherDoggie/status/1984852266105401480', 'https://x.com/83thegame', 'https://x.com/NYPD83Pct', 'https://x.com/BorisKodjoe/status/1877874266571440370', 'https://x.com/thevinceneil/status/1299884839202705412', 'https://x.com/patrickbetdavid/status/650570310643806208', 'https://x.com/marcorubio/status/189902

KeyboardInterrupt: 